<a href="https://colab.research.google.com/github/Harshitha82/UE25CS645BC2_PES1PG25CS082_Fashion_MNIST_CNN/blob/main/UE25CS645BC2_PES1PG25CS082.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [38]:
import numpy as np
from tensorflow.keras.datasets import fashion_mnist

In [39]:
(train_images, train_labels), (test_images, test_labels) = fashion_mnist.load_data()

In [40]:
train_images = train_images / 255.0
test_images = test_images / 255.0
print("Train Shape:", train_images.shape)
print("Test Shape:", test_images.shape)

Train Shape: (60000, 28, 28)
Test Shape: (10000, 28, 28)


In [41]:
class ConvLayer:

    def __init__(self, num_filters, filter_size):

        self.num_filters = num_filters
        self.filter_size = filter_size

        self.filters = np.random.randn(
            num_filters,
            filter_size,
            filter_size
        ) / (filter_size * filter_size)

    def forward(self, input):

        self.input = input

        h, w = input.shape

        output = np.zeros((
            self.num_filters,
            h - self.filter_size + 1,
            w - self.filter_size + 1
        ))

        for f in range(self.num_filters):

            for i in range(h - self.filter_size + 1):

                for j in range(w - self.filter_size + 1):

                    region = input[
                        i:i+self.filter_size,
                        j:j+self.filter_size
                    ]

                    output[f, i, j] = np.sum(
                        region * self.filters[f]
                    )

        return output

    def backward(self, d_out, learning_rate):

        d_filters = np.zeros(self.filters.shape)

        for f in range(self.num_filters):

            for i in range(d_out.shape[1]):

                for j in range(d_out.shape[2]):

                    region = self.input[
                        i:i+self.filter_size,
                        j:j+self.filter_size
                    ]

                    d_filters[f] += d_out[f, i, j] * region
        self.filters -= learning_rate * d_filters


In [42]:
class MaxPool:

    def __init__(self, size):

        self.size = size

    def forward(self, input):

        self.input = input

        num_filters, h, w = input.shape

        output = np.zeros((
            num_filters,
            h // 2,
            w // 2
        ))

        for f in range(num_filters):

            for i in range(h // 2):

                for j in range(w // 2):

                    region = input[
                        f,
                        i*2:i*2+2,
                        j*2:j*2+2
                    ]

                    output[f, i, j] = np.max(region)

        return output

In [43]:
def flatten(input):

    return input.flatten()

In [44]:
class Dense:

    def __init__(self, input_len, output_len):

        self.weights = np.random.randn(
            input_len,
            output_len
        ) / input_len

        self.bias = np.zeros(output_len)

    def forward(self, input):

        self.input = input

        return np.dot(input, self.weights) + self.bias

    def backward(self, d_out, learning_rate):

        d_weights = np.outer(self.input, d_out)

        d_input = np.dot(self.weights, d_out)

        self.weights -= learning_rate * d_weights

        self.bias -= learning_rate * d_out

        return d_input

In [45]:
def softmax(x):

    exp = np.exp(x - np.max(x))

    return exp / np.sum(exp)

In [46]:
def cross_entropy(probs, label):

    return -np.log(probs[label])


In [47]:
conv = ConvLayer(num_filters=8, filter_size=3)

dense = Dense(26 * 26 * 8, 10)

In [48]:
def forward(image, label):

    out = conv.forward(image)

    out = flatten(out)

    out = dense.forward(out)

    probs = softmax(out)

    loss = cross_entropy(probs, label)

    return probs, loss

In [49]:
def train(image, label, learning_rate=0.005):

    # forward pass
    probs, loss = forward(image, label)

    # gradient
    gradient = probs.copy()

    gradient[label] -= 1

    # dense backward
    grad_back = dense.backward(
        gradient,
        learning_rate
    )

    # reshape
    grad_back = grad_back.reshape(8, 26, 26)

    # conv backward
    conv.backward(
        grad_back,
        learning_rate
    )

    return loss

In [50]:
print("\nTraining Started...\n")

for epoch in range(3):

    print("Epoch:", epoch + 1)

    total_loss = 0

    # training on first 1000 images
    for i in range(1000):

        image = train_images[i]

        label = train_labels[i]

        loss = train(image, label)

        total_loss += loss

        if i % 100 == 0:

            print(
                "Step:",
                i,
                "Loss:",
                round(loss, 3)
            )

    print("Average Loss:", total_loss / 1000)

    print()


Training Started...

Epoch: 1
Step: 0 Loss: 2.305
Step: 100 Loss: 2.451
Step: 200 Loss: 2.695
Step: 300 Loss: 1.893
Step: 400 Loss: 1.567
Step: 500 Loss: 3.515
Step: 600 Loss: 0.877
Step: 700 Loss: 2.618
Step: 800 Loss: 0.102
Step: 900 Loss: 0.337
Average Loss: 1.1913518100748735

Epoch: 2
Step: 0 Loss: 0.1
Step: 100 Loss: 0.055
Step: 200 Loss: 1.025
Step: 300 Loss: 1.579
Step: 400 Loss: 0.571
Step: 500 Loss: 2.048
Step: 600 Loss: 0.586
Step: 700 Loss: 2.577
Step: 800 Loss: 0.049
Step: 900 Loss: 0.044
Average Loss: 0.690315878341573

Epoch: 3
Step: 0 Loss: 0.023
Step: 100 Loss: 0.028
Step: 200 Loss: 0.599
Step: 300 Loss: 1.55
Step: 400 Loss: 0.291
Step: 500 Loss: 1.101
Step: 600 Loss: 0.439
Step: 700 Loss: 2.281
Step: 800 Loss: 0.02
Step: 900 Loss: 0.029
Average Loss: 0.5411190958723024



In [52]:
correct = 0

for i in range(1000):

    image = test_images[i]

    label = test_labels[i]

    probs, loss = forward(image, label)

    prediction = np.argmax(probs)

    if prediction == label:

        correct += 1

accuracy = (correct / 1000) * 100

print("\nFinal Accuracy: %.2f%%" % accuracy)


Final Accuracy: 75.50%
